# Microsoft India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** gcsservices.careers.microsoft.com (GCS Services API)

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:57:03
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Microsoft"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Microsoft/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("MICROSOFT INDIA JOB SCRAPER")
print("Source: gcsservices.careers.microsoft.com/search/api/v1/search")
print("=" * 60)

microsoft_jobs = []
session = get_session()
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

# Microsoft uses the GCS Services API (confirmed Mar 2026)
api_url = "https://gcsservices.careers.microsoft.com/search/api/v1/search"

print("  Using Microsoft GCS Services API...")
page = 1
max_retries = 3

while len(microsoft_jobs) < 1000:
    params = {
        "l": "en_us",
        "pg": page,
        "pgSz": 20,
        "o": "Relevance",
        "flt": "true",
        "loc": "India",
    }

    success = False
    for attempt in range(max_retries):
        try:
            resp = session.get(api_url, params=params, timeout=30)
            if resp.status_code == 200:
                success = True
                break
            elif resp.status_code == 502:
                print(f"  API returned 502 (attempt {attempt+1}), retrying...")
                time.sleep(5)
            else:
                print(f"  API returned {resp.status_code}")
                break
        except Exception as e:
            print(f"  Request error (attempt {attempt+1}): {e}")
            time.sleep(3)

    if not success:
        break

    try:
        data = resp.json()
        result = data.get("operationResult", {}).get("result", {})
        jobs_list = result.get("jobs", [])
        total = result.get("totalJobs", 0)

        if not jobs_list:
            break

        print(f"  Page {page}: {len(jobs_list)} jobs (total available: {total})")

        for job in jobs_list:
            props = job.get("properties", job)
            title = props.get("title", job.get("title", ""))
            loc = props.get("primaryLocation", props.get("location", "India"))
            city = loc.split(",")[0].strip() if loc else "India"
            jd = props.get("description", "")

            microsoft_jobs.append({
                "job_id": str(props.get("jobId", job.get("jobId", len(microsoft_jobs)))),
                "title": title,
                "company_name": "Microsoft",
                "raw_jd_text": html_to_text(jd),
                "location_city": city,
                "industry": "Technology",
                "date_posted": str(props.get("datePosted", props.get("postingDate",
                    datetime.now().strftime("%Y-%m-%d"))))[:10],
                "is_active": True,
                "job_url": f"https://jobs.careers.microsoft.com/global/en/job/{props.get('jobId', '')}",
                "business_unit": props.get("category", props.get("discipline", "")),
                "source_platform": "Microsoft GCS API",
            })

        if len(jobs_list) < 20 or page * 20 >= total:
            break
        page += 1
        time.sleep(random.uniform(0.5, 1.5))

    except Exception as e:
        print(f"  Parse error: {e}")
        break

# Selenium fallback if API fails
if len(microsoft_jobs) < 5:
    print("\n  API failed, trying Selenium on jobs.careers.microsoft.com...")
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    driver = setup_selenium()
    try:
        driver.get("https://jobs.careers.microsoft.com/global/en/search?lc=India&l=en_us&pg=1&pgSz=20&o=Relevance")
        time.sleep(10)

        for pg in range(5):
            soup = BeautifulSoup(driver.page_source, "lxml")
            cards = soup.select("[class*='ms-List-cell'], [class*='job-card'], [role='listitem']")
            if not cards:
                cards = soup.select("a[href*='/job/'], [data-automation-id*='job']")

            for card in cards:
                title_el = card.select_one("h2, h3, [class*='title'], a")
                title = title_el.get_text(strip=True) if title_el else ""
                loc_el = card.select_one("[class*='location'], [class*='city']")
                loc = loc_el.get_text(strip=True) if loc_el else "India"

                if title and len(title) > 3 and title not in [j["title"] for j in microsoft_jobs]:
                    microsoft_jobs.append({
                        "job_id": str(len(microsoft_jobs)),
                        "title": title,
                        "company_name": "Microsoft",
                        "raw_jd_text": card.get_text(" ", strip=True),
                        "location_city": loc.split(",")[0].strip(),
                        "industry": "Technology",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "job_url": "",
                        "business_unit": "",
                        "source_platform": "Selenium",
                    })

            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Next'], [class*='next']")
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(4)
            except:
                break
    except Exception as e:
        print(f"  Selenium error: {e}")
    finally:
        driver.quit()

print(f"Total Microsoft India jobs: {len(microsoft_jobs)}")


MICROSOFT INDIA JOB SCRAPER
Source: gcsservices.careers.microsoft.com/search/api/v1/search
  Using Microsoft GCS Services API...


  API returned 502 (attempt 1), retrying...


  API returned 502 (attempt 2), retrying...


  API returned 502 (attempt 3), retrying...



  API failed, trying Selenium on jobs.careers.microsoft.com...


Total Microsoft India jobs: 0


In [5]:
df_microsoft = save_results(microsoft_jobs, "Microsoft", OUTPUT_DIR)
if df_microsoft is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_microsoft.columns]
    print(df_microsoft[cols].head(10).to_string())


  [WARN] No jobs found for Microsoft
